# Real Estate Price Model Training

This notebook trains the same ExtraTrees pipeline used for the R² = 0.6684 run.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


In [ ]:
candidate_paths = [
    Path(r"Data\Real_Estate_Dataset_Clean_Nairobi_rule_imputed.csv"),
    Path(r"Real_Estate_Dataset_Clean_Nairobi_rule_imputed.csv"),
]
data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find Real_Estate_Dataset_Clean_Nairobi_rule_imputed.csv")

df = pd.read_csv(data_path)

for c in [
    "No. bedrooms",
    "No. Bathrooms",
    "Floor area Square Meters",
    "Advertised Price",
    "Year property was advertised",
    "Land Size where property is built on (Acres)",
    "Year of house construction completion",
]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

for c in ["Type of Resudential Property", "Sub County Located", "County Located"]:
    if c in df.columns:
        df[c] = (
            df[c]
            .astype(str)
            .str.strip()
            .replace({"": np.nan, "nan": np.nan, "None": np.nan})
            .str.upper()
        )

required_core = [
    "Type of Resudential Property",
    "No. bedrooms",
    "Sub County Located",
    "Floor area Square Meters",
    "Advertised Price",
]
df = df.dropna(subset=required_core).copy()
df.shape


In [ ]:
# Keep same feature space as the run that produced R² ~= 0.6684
exclude_cols = {"Advertised Price", "Bathrooms Imputed Rule", "Floor Area Imputed Rule"}
feature_cols = [c for c in df.columns if c not in exclude_cols]

X = df[feature_cols].copy()
y = df["Advertised Price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median"))]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])

model = ExtraTreesRegressor(
    n_estimators=2000,
    max_depth=30,
    min_samples_leaf=1,
    max_features=1.0,
    random_state=42,
    n_jobs=-1,
)

reg = TransformedTargetRegressor(
    regressor=Pipeline([("pre", pre), ("m", model)]),
    func=np.log1p,
    inverse_func=np.expm1,
)

reg.fit(X_train, y_train)
pred = reg.predict(X_test)

r2 = r2_score(y_test, pred)
mae = mean_absolute_error(y_test, pred)
rmse = mean_squared_error(y_test, pred) ** 0.5

print(f"Rows used: {len(df)}")
print(f"R2: {r2:.4f}")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
